- **Gradient Boosting (often called GBD or GBDT - Gradient Boosted Decision Trees)** is an advanced sequential ensemble technique.
- **Gradient Boosting** is an algorithm that constructs a strong predictive ensemble by **sequentially fitting weak decision trees *(Boosting)* to the negative gradient (slope) of any custom loss function.**

# Gradient Boosting Regressor

## The Core Intuition

1. **The Base Guess:** The algorithm starts by making a single, baseline prediction for the entire dataset (usually just the average value of your target variable).

$$F_0(x) = \bar{y}$$

2. **Calculate the Error:** It calculates how far off this base guess is for every single row. This difference (Actual Value minus Predicted Value) is called a **Residual.**

$$r_i = y_i - F_{m-1}(x_i)$$

**Note:** In Gradient Boosting Regressor we use Half Squarred Error Loss. The derivative of which becomes y - F.

3. **Train on Errors:** The next decision tree is not trained to predict the final target. It is trained specifically to predict those residuals (the errors).

$$h_m(x) = \begin{cases} \text{mean}(r \text{ where } x \le T) & \text{if } x \le T \\ \text{mean}(r \text{ where } x > T) & \text{if } x > T \end{cases}$$

4. **Update the Prediction:** The algorithm adds the new tree's prediction to the running total, bringing the overall guess closer to the true value.

$$F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x)$$

5. **Repeat:** The process repeats for many rounds. Each new tree isolates and chips away at the remaining errors left behind by the combined group of previous trees.

$$F_M(x) = \bar{y} + \nu \cdot h_1(x) + \nu \cdot h_2(x) + \dots + \nu \cdot h_M(x)$$

## Mathematical Example

1. **The Dataset**
    We want to predict a house price (Y) based on its size (X).
    
    |Row|Feature (X)|True Target (Y)|
    |---|---|---|
    |1|1.0|10|
    |2|2.0|20|
    |3|3.0|60|

    **Base Guess** (Mean value of Y): For squared error loss, this base guess is simply the average value of Y:
    $$F_{0}(x)=\frac{10+20+60}{3}=\frac{90}{3}=\mathbf{30}$$
    $$\hat y = [30,30,30]$$

2. **Calcuate the Error**:
    $$\text{Residual} = Y - F₀(x)$$
    - Row 1 Residual : 10 - 30  = -20
    - Row 2 Residual : 20 - 30  = -10
    - Row 3 Residual : 60 - 30  = +30

3. **Train Tree to Predict the Residuals**:
    We now train a decision tree stump (h₁(x)). Crucially, the target column for this tree is the Residual column, not the original Y.
    |Row|Feature (X)|Target for Tree 1 (Residual)|
    |---|---|---|
    |1|1.0|-20|
    |2|2.0|-10|
    |3|3.0|+30|

    Let's say the tree selects a split threshold at X ≤ 2.5:
    - Left Branch: Contains Row 1 and Row 2. The tree predicts the average of their residuals: $\frac{-20 + (-10)}{2} = \mathbf{-15}$
    - Right Branch (X > 2.5): Contains Row 3. The tree predicts its residual: $\mathbf{+30}$.

    Tree 1 Predictions $(h_1(x))$: [-15,-15,+30]

4. **Update the Running Ensemble Prediction (F₁(x))**
    We update our overall prediction by adding the new tree's output to our previous guess.
    To prevent overfitting, we scale the new tree's contribution by our Learning Rate (ν = 0.1):
    $$F_{1}(x)=F_{0}(x)+(\nu \cdot h_{1}(x))$$

    - Row 1 New Guess: $30 + (0.1 \cdot (-15)) = 30 - 1.5 = \mathbf{28.5}$ (Brought closer to 10)
    - Row 2 New Guess: $30 + (0.1 \cdot (-15)) = 30 - 1.5 = \mathbf{28.5}$ (Brought closer to 20)
    - Row 3 New Guess: $30 + (0.1 \cdot (+30)) = 30 + 3.0 = \mathbf{33.0}$ (Brought closer to 60)

5. **Calculate the Next Residuals for the next round**
    Round 1 is complete. To prepare for Round 2, we calculate a brand-new set of residuals based on our updated guesses: New Residual = Y - F₁(x).
    - Row 1 New Residual: $10 - 28.5 = \mathbf{-18.5}$
    - Row 2 New Residual: $20 - 28.5 = \mathbf{-8.5}$
    - Row 3 New Residual: $60 - 33.0 = \mathbf{+27.0}$

    Now Repeat the same steps again ....


6. **Prediction**
    When a brand-new, completely unknown test sample arrives, it does not have a \(Y\) value, so we cannot calculate residuals for it. Instead, we pass its features through the exact chain of components we built:
    $$\text{Final\ Prediction}=\text{Base\ Mean}+\nu \cdot \text{Tree}_{1}(x)+\nu \cdot \text{Tree}_{2}(x)+\dots +\nu \cdot \text{Tree}_{M}(x)$$





## Python Code

In [9]:
import numpy as np


class SimpleDecisionStump:
    """Decision stump built specifically to predict mean residuals."""

    def __init__(self):
        self.feature_idx = 0
        self.threshold = None
        self.left_mean = 0.0
        self.right_mean = 0.0

    def fit(self, X, residuals):
        n_samples, n_features = X.shape
        best_sse = float("inf")

        for f_idx in range(n_features):
            X_col = X[:, f_idx]
            sorted_unique = np.sort(np.unique(X_col))

            if len(sorted_unique) <= 1:
                continue

            # Candidate midpoint thresholds
            thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

            for t in thresholds:
                left_mask = X_col <= t
                right_mask = ~left_mask

                if not np.any(left_mask) or not np.any(right_mask):
                    continue

                # Mean of residuals in each branch
                left_val = np.mean(residuals[left_mask])
                right_val = np.mean(residuals[right_mask])

                # Calculate Sum of Squared Errors (SSE)
                preds = np.where(left_mask, left_val, right_val)
                sse = np.sum((residuals - preds) ** 2)

                if sse < best_sse:
                    best_sse = sse
                    self.feature_idx = f_idx
                    self.threshold = t
                    self.left_mean = left_val
                    self.right_mean = right_val

    def predict(self, X):
        return np.where(
            X[:, self.feature_idx] <= self.threshold,
            self.left_mean,
            self.right_mean,
        )


class SimpleGradientBoostingRegressor:
    """GBDT using simplified MSE residual rules."""

    def __init__(self, n_estimators=3, learning_rate=0.1):
        self.M = n_estimators
        self.nu = learning_rate
        self.y_bar = 0.0
        self.trees = []

    def fit(self, X, y):
        # Step 1: Base guess is the mean of y
        self.y_bar = np.mean(y)
        F = np.full_like(y, self.y_bar, dtype=np.float64)

        print("--- GBDT TRAINING LOOP ---")
        print(f"Base Guess F0(x) = {self.y_bar:.3f}\n")

        for m in range(self.M):
            # Step 2: Compute Residuals (r = y - F)
            residuals = y - F

            # Step 3: Train Tree Stump on Residuals
            tree = SimpleDecisionStump()
            tree.fit(X, residuals)

            # Step 4: Tree prediction h_m(x)
            h_m = tree.predict(X)

            # Step 5: Update F_m(x) = F_{m-1}(x) + nu * h_m(x)
            F = F + self.nu * h_m

            self.trees.append(tree)

            print(
                f"Round {m+1}: Split @ X[{tree.feature_idx}] <= {tree.threshold:.2f}"
            )
            print(f"   Left Mean = {tree.left_mean:.3f} | Right Mean = {tree.right_mean:.3f}")
            print(f"   Current Predictions F_{m+1}(x): {np.round(F, 3)}")

    def predict(self, X_new):
        # Step 6: Final prediction = y_bar + nu * tree1(x) + nu * tree2(x) ...
        predictions = np.full(X_new.shape[0], self.y_bar, dtype=np.float64)

        for tree in self.trees:
            predictions += self.nu * tree.predict(X_new)

        return predictions


if __name__ == "__main__":
    X_train = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
    y_train = np.array([2.0, 5.0, 7.0, 4.0, 8.0])

    model = SimpleGradientBoostingRegressor(n_estimators=5, learning_rate=0.1)
    model.fit(X_train, y_train)

    X_test = np.array([[2.5], [4.2]])
    preds = model.predict(X_test)

    print("\n--- INFERENCE RESULTS ---")
    for x_val, pred in zip(X_test.ravel(), preds):
        print(f"Final Prediction for X = {x_val}: {pred:.3f}")

--- GBDT TRAINING LOOP ---
Base Guess F0(x) = 5.200

Round 1: Split @ X[0] <= 1.50
   Left Mean = -3.200 | Right Mean = 0.800
   Current Predictions F_1(x): [4.88 5.28 5.28 5.28 5.28]
Round 2: Split @ X[0] <= 1.50
   Left Mean = -2.880 | Right Mean = 0.720
   Current Predictions F_2(x): [4.592 5.352 5.352 5.352 5.352]
Round 3: Split @ X[0] <= 4.50
   Left Mean = -0.662 | Right Mean = 2.648
   Current Predictions F_3(x): [4.526 5.286 5.286 5.286 5.617]
Round 4: Split @ X[0] <= 1.50
   Left Mean = -2.526 | Right Mean = 0.631
   Current Predictions F_4(x): [4.273 5.349 5.349 5.349 5.68 ]
Round 5: Split @ X[0] <= 4.50
   Left Mean = -0.580 | Right Mean = 2.320
   Current Predictions F_5(x): [4.215 5.291 5.291 5.291 5.912]

--- INFERENCE RESULTS ---
Final Prediction for X = 2.5: 5.291
Final Prediction for X = 4.2: 5.291


# Gradient Boosting Classifier (GBC)

Instead of fitting trees directly to continuous residuals ($y - F$), GBC fits trees to probability residuals using Log-Loss (Binary Cross-Entropy).



## The Core Intuition

1. **The Base Guess**:
    In binary classification, this starting guess is the Log-Odds of the positive class (1).
    $$F_{0}(x)=\ln \left(\frac{\text{Count\ of\ 1s}}{\text{Count\ of\ 0s}}\right)$$

2. **Calculate the Error:** 
    It converts the current log-odds prediction into a probability (p) using the sigmoid function. It then calculates how far off that probability is from the true binary label 0 or 1. This difference is called a **Pseudo-Residual**.
    $$r_{m,i}=y_{i}-p_{m-1}(x_{i})\quad \text{where}\quad p_{m-1}(x_{i})=\frac{1}{1+e^{-F_{m-1}(x_{i})}}$$

3. **Train on Errors:**
    The next decision tree is trained to predict those pseudo-residuals using variance reduction.
    The algorithm must run a Taylor series approximation to transform the output $\gamma \$ of each leaf node j.
    $$\gamma =\frac{\sum \text{Residuals}}{\sum p(1-p)}$$

4. **Update the Prediction:**
    The algorithm adds the transformed log-odds tree outputs to the running log-odds total, scaling the new tree's contribution by a Learning Rate $(\nu )$ to prevent overfitting.
    $$F_{m}(x)=F_{m-1}(x)+\nu \cdot \gamma _{m}(x)$$

5. **Repeat:** The process repeats for many rounds. Once all \(M\) trees are built, the final log-odds summation is passed through a final sigmoid filter to convert the ensemble score into an explicit **Classification Probability.**

$$P(Y=1|x)=\frac{1}{1+e^{-F_{M}(x)}}\quad \text{where}\quad F_{M}(x)=F_{0}(x)+\nu \cdot \gamma _{1}(x)+\dots +\nu \cdot \gamma _{M}(x)$$





## Mathematical Example

**Dataset**
- $X = [1, 2, 3, 4]$
- $y = [0, 0, 1, 1]$
- Hyperparameters: Learning rate $\nu = 0.2$, Loss function = Binary Cross-Entropy (Log-Loss)

1. **Base Prediction $F_0(x)$**
    $$F_0(x) = \ln\left(\frac{\text{Count of } y=1}{\text{Count of } y=0}\right) = \ln\left(\frac{2}{2}\right) = \ln(1) = 0.000$$
    $$p_i^{(0)} = \sigma(F_0) = \frac{1}{1 + e^{-0}} = 0.500 \quad \text{for all } i$$
    Initial probability vector: $p^{(0)} = [0.500, 0.500, 0.500, 0.500]$

2. **Boosting Round m=1**

    
    1. **Calculate Pseudo-Residuals ($r_{i1} = y_i - p_i^{(0)}$)**

    |i|$x_i$|​$y_i$|​$p_i^{(0)}$​|Residual $r_{i1}​=y_i​−p_i^{(0)}$|​Hessian $h_{i1​}=pi^{(0)}​(1−p_i^{(0)}​)$
    |---|---|---|---|---|---|
    |1|1|0|0.500|$-0.500$|$0.500 \times 0.500 = \mathbf{0.250}$|
    |2|2|0|0.500|$-0.500$|$0.500 \times 0.500 = \mathbf{0.250}$|
    |3|3|1|0.500|$+0.500$|$0.500 \times 0.500 = \mathbf{0.250}$|
    |4|4|1|0.500|$+0.500$|$0.500 \times 0.500 = \mathbf{0.250}$|

    Target vector for Tree 1 becomes: $r_1 = [-0.500, -0.500, +0.500, +0.500]$.

    2. **Fit Weak Regression Tree Stump on $r_1$**
    
    Let threshold is $x \le 2.5$:
    - Left Leaf ($R_{11}$): $x \in \{1, 2\}$
    - Right Leaf ($R_{21}$): $x \in \{3, 4\}$

    3. **Compute Transformed Leaf Outputs ($\gamma_{j1}$)**

    Using the Newton-Raphson leaf approximation $\gamma = \frac{\sum r_i}{\sum p_i(1 - p_i)}$:

    - Left Leaf Output ($\gamma_{11}$):
        $$\gamma_{11} = \frac{(-0.500) + (-0.500)}{0.250 + 0.250} = \frac{-1.000}{0.500} = \mathbf{-2.000}$$
    
    - Right Leaf Output ($\gamma_{21}$):
        $$\gamma_{21} = \frac{(+0.500) + (+0.500)}{0.250 + 0.250} = \frac{+1.000}{0.500} = \mathbf{+2.000}$$Tree output in log-odds space:$$h_1(X) = [-2.000, -2.000, +2.000, +2.000]$$

    4. **Update Ensemble Log-Odds $F_1(x)$ & Probabilities $p^{(1)}$**
        $$F_1(x) = F_0(x) + \nu \cdot h_1(x) = 0.000 + 0.2 \cdot h_1(x)$$
        - For $x \in \{1, 2\}$:
            - $F_1(x) = 0.000 + 0.2(-2.000) = \mathbf{-0.400}$
            - $p^{(1)} = \sigma(-0.400) = \frac{1}{1 + e^{0.400}} \approx \mathbf{0.401}$
        - For $x \in \{3, 4\}$:
            - $F_1(x) = 0.000 + 0.2(+2.000) = \mathbf{+0.400}$
            - $p^{(1)} = \sigma(+0.400) = \frac{1}{1 + e^{-0.400}} \approx \mathbf{0.599}$

3. Repeat the steps
4. Summary

    i|$x_i|​Actual Class $(y_i​)$|Round 0 $(p^{(0)})$|Round 1 $(p^{(1)})$|Round 2 $(p^{(2)})$|Direction|
    |---|---|---|---|---|---|---|
    |1|1|0|0.500|0.401|0.324|Dropping toward 0|
    |2|2|0|0.500|0.401|0.324|Dropping toward 0|
    |3|3|1|0.500|0.599|0.676|Rising toward 1|
    |4|4|1|0.500|0.599|0.676|Rising toward 1|


## Python Code

In [1]:
import numpy as np


def sigmoid(z):
    """Numerically stable Sigmoid function converting log-odds to probabilities."""
    return 1.0 / (1.0 + np.exp(-z))


class GBCDecisionStump:
    """Decision stump optimized for classification using Newton-Raphson leaf steps."""

    def __init__(self):
        self.feature_idx = 0
        self.threshold = None
        self.left_gamma = 0.0
        self.right_gamma = 0.0

    def fit(self, X, residuals, hessians):
        n_samples, n_features = X.shape
        best_sse = float("inf")

        for f_idx in range(n_features):
            X_col = X[:, f_idx]
            sorted_unique = np.sort(np.unique(X_col))

            if len(sorted_unique) <= 1:
                continue

            # Candidate midpoint split thresholds
            thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

            for t in thresholds:
                left_mask = X_col <= t
                right_mask = ~left_mask

                if not np.any(left_mask) or not np.any(right_mask):
                    continue

                # Newton-Raphson leaf approximation: gamma = sum(r) / sum(h)
                left_gamma = np.sum(residuals[left_mask]) / (
                    np.sum(hessians[left_mask]) + 1e-10
                )
                right_gamma = np.sum(residuals[right_mask]) / (
                    np.sum(hessians[right_mask]) + 1e-10
                )

                # Variance reduction check on residuals
                preds = np.where(left_mask, left_gamma, right_gamma)
                sse = np.sum((residuals - preds) ** 2)

                if sse < best_sse:
                    best_sse = sse
                    self.feature_idx = f_idx
                    self.threshold = t
                    self.left_gamma = left_gamma
                    self.right_gamma = right_gamma

    def predict(self, X):
        return np.where(
            X[:, self.feature_idx] <= self.threshold,
            self.left_gamma,
            self.right_gamma,
        )


class GradientBoostingClassifierFromScratch:
    """Binary Gradient Boosting Classifier using Binary Cross-Entropy (Log-Loss)."""

    def __init__(self, n_estimators=2, learning_rate=0.2):
        self.M = n_estimators
        self.nu = learning_rate
        self.F0 = 0.0
        self.trees = []

    def fit(self, X, y):
        # Step 1: Base prediction F0 = ln(count(1) / count(0))
        pos_count = np.sum(y == 1)
        neg_count = np.sum(y == 0)
        self.F0 = np.log((pos_count + 1e-10) / (neg_count + 1e-10))

        F = np.full(X.shape[0], self.F0, dtype=np.float64)

        print("--- GRADIENT BOOSTING CLASSIFIER TRAINING LOOP ---")
        print(f"Base Log-Odds F0 = {self.F0:.3f}")
        print(f"Initial Probabilities p(0) = {np.round(sigmoid(F), 3)}\n")

        for m in range(self.M):
            p = sigmoid(F)

            # Step 2: Compute Pseudo-Residuals (r_i = y_i - p_i) and Hessians (h_i = p_i * (1 - p_i))
            residuals = y - p
            hessians = p * (1.0 - p)

            # Step 3: Train Tree Stump on Residuals
            tree = GBCDecisionStump()
            tree.fit(X, residuals, hessians)

            # Step 4: Predict Transformed Log-Odds Leaf Outputs
            gamma_m = tree.predict(X)

            # Step 5: Update Ensemble Log-Odds F_m(x) = F_{m-1}(x) + nu * gamma_m(x)
            F += self.nu * gamma_m

            self.trees.append(tree)

            print(f"Round {m+1}: Split @ X[{tree.feature_idx}] <= {tree.threshold:.2f}")
            print(f"   Left Gamma  = {tree.left_gamma:.3f} | Right Gamma = {tree.right_gamma:.3f}")
            print(f"   Updated Log-Odds F_{m+1}(x) = {np.round(F, 3)}")
            print(f"   Updated Probabilities p({m+1}) = {np.round(sigmoid(F), 3)}\n")

    def predict_proba(self, X_new):
        F = np.full(X_new.shape[0], self.F0, dtype=np.float64)
        for tree in self.trees:
            F += self.nu * tree.predict(X_new)
        return sigmoid(F)

    def predict(self, X_new):
        return (self.predict_proba(X_new) >= 0.5).astype(int)


if __name__ == "__main__":
    # Toy dataset matching the mathematical walkthrough exactly
    X_train = np.array([[1.0], [2.0], [3.0], [4.0]])
    y_train = np.array([0, 0, 1, 1])

    gbc = GradientBoostingClassifierFromScratch(n_estimators=2, learning_rate=0.2)
    gbc.fit(X_train, y_train)

    X_test = np.array([[1.5], [3.5]])
    probs = gbc.predict_proba(X_test)
    preds = gbc.predict(X_test)

    print("--- INFERENCE RESULTS ---")
    for x_val, p_val, c_val in zip(X_test.ravel(), probs, preds):
        print(f"X = {x_val}: Predicted Prob = {p_val:.3f} -> Predicted Class = {c_val}")

--- GRADIENT BOOSTING CLASSIFIER TRAINING LOOP ---
Base Log-Odds F0 = 0.000
Initial Probabilities p(0) = [0.5 0.5 0.5 0.5]

Round 1: Split @ X[0] <= 1.50
   Left Gamma  = -2.000 | Right Gamma = 0.667
   Updated Log-Odds F_1(x) = [-0.4    0.133  0.133  0.133]
   Updated Probabilities p(1) = [0.401 0.533 0.533 0.533]

Round 2: Split @ X[0] <= 1.50
   Left Gamma  = -1.670 | Right Gamma = 0.536
   Updated Log-Odds F_2(x) = [-0.734  0.241  0.241  0.241]
   Updated Probabilities p(2) = [0.324 0.56  0.56  0.56 ]

--- INFERENCE RESULTS ---
X = 1.5: Predicted Prob = 0.324 -> Predicted Class = 0
X = 3.5: Predicted Prob = 0.560 -> Predicted Class = 1


# Multi-Class GBC

## Binary vs. Multi-Class GBC

|Feature|Binary Classification (K=2)|Multi-Class Classification (K>2)|
|---|---|---|
|Trees per Round|$1$ Tree|$K$ Trees (1 for each class)|
|Output Space|Single Scalar Log-Odds $F(x)$|Vector of $K$ Log-Odds $[F^{(0)}, \dots, F^{(K-1)}]$|
|Probability Function|Sigmoid $\sigma(F(x))$|Softmax $\text{softmax}(F(x))$|
|Residual Formula|$y_i - p_i$|$y_i^{(k)} - p_i^{(k)}$|
|Decision Rule|$p \ge 0.5 \implies 1$|$\arg\max_k (p^{(k)})$|

## Step-by-Step Multi-Class Algorithm

1. **Initialize Base Predictions $F_0^{(k)}(x)$**
    $$F_0^{(k)}(x) = 0.0 \quad \text{for } k = 0, 1, \dots, K-1$$
    This yields uniform initial probabilities: $p^{(k)} = \frac{1}{K}$.

2. **Boosting Loop (For $m = 1 \dots M$)**
    1. Convert current ensemble scores $F_{m-1}(x)$ to probabilities $p_i^{(k)}$ using Softmax.
    2. For each class $k \in \{0, \dots, K-1\}$:
        - Compute class $k$'s pseudo-residual: $r_{i, m}^{(k)} = y_i^{(k)} - p_i^{(k)}$
        - Fit a weak tree stump $h_{m}^{(k)}(x)$ to predict $r_{i, m}^{(k)}$.
        - Compute Newton-Raphson transformed leaf values ($\gamma_{j, m}^{(k)}$):
        $$\gamma_j^{(k)} = \frac{\sum_{x_i \in R_j} r_{i}^{(k)}}{\sum_{x_i \in R_j} p_i^{(k)}(1 - p_i^{(k)})}$$
        - Update class $k$'s log-odds score:
        $$F_m^{(k)}(x) = F_{m-1}^{(k)}(x) + \nu \cdot \gamma_{m}^{(k)}(x)$$

3. **Final Inference**
    1. Pass the final log-odds vector $F_M(x)$ through Softmax to retrieve class probabilities.
    2. The predicted class is simply the index with the maximum probability:$$\hat{y} = \arg\max_k \left( p^{(k)} \right)$$

## Python Implementation

In [13]:
import numpy as np


def softmax(z):
    """Numerically stable Softmax transformation converting log-odds to probabilities."""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


class MultiClassGBCStump:
    """Decision stump optimized for single-class multinomial log-odds residuals."""

    def __init__(self):
        self.feature_idx = 0
        self.threshold = None
        self.left_gamma = 0.0
        self.right_gamma = 0.0

    def fit(self, X, residuals, hessians):
        n_samples, n_features = X.shape
        best_sse = float("inf")

        for f_idx in range(n_features):
            X_col = X[:, f_idx]
            sorted_unique = np.sort(np.unique(X_col))

            if len(sorted_unique) <= 1:
                continue

            thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

            for t in thresholds:
                left_mask = X_col <= t
                right_mask = ~left_mask

                if not np.any(left_mask) or not np.any(right_mask):
                    continue

                # 1. FIND THE SPLIT USING RAW RESIDUAL MEANS
                left_mean = np.mean(residuals[left_mask])
                right_mean = np.mean(residuals[right_mask])

                preds = np.where(left_mask, left_mean, right_mean)
                sse = np.sum((residuals - preds) ** 2)

                # 2. IF IT'S THE BEST SPLIT, LOCK IT IN AND CALCULATE GAMMAS
                if sse < best_sse:
                    best_sse = sse
                    self.feature_idx = f_idx
                    self.threshold = t
                    
                    # Convert the winning branches to log-odds space using the Hessians
                    self.left_gamma = np.sum(residuals[left_mask]) / (
                        np.sum(hessians[left_mask]) + 1e-10
                    )
                    self.right_gamma = np.sum(residuals[right_mask]) / (
                        np.sum(hessians[right_mask]) + 1e-10
                    )


    def predict(self, X):
        return np.where(
            X[:, self.feature_idx] <= self.threshold,
            self.left_gamma,
            self.right_gamma,
        )


class MultiClassGradientBoostingClassifier:
    """Multi-Class GBDT Classifier using Multinomial Log-Loss & Softmax."""

    def __init__(self, n_classes=3, n_estimators=3, learning_rate=0.2):
        self.K = n_classes
        self.M = n_estimators
        self.nu = learning_rate
        # Array of trees: shape (M_rounds, K_classes)
        self.trees = []

    def fit(self, X, y):
        n_samples = X.shape[0]

        # One-hot encode targets y into (N, K) matrix
        Y_onehot = np.zeros((n_samples, self.K))
        Y_onehot[np.arange(n_samples), y] = 1.0

        # Initial Log-Odds matrix F0 of zeros (N, K)
        F = np.zeros((n_samples, self.K), dtype=np.float64)

        print("--- MULTI-CLASS GBC TRAINING LOOP ---")
        print(f"Number of Classes K = {self.K}")
        print(f"Initial Probabilities p(0):\n{np.round(softmax(F), 3)}\n")

        for m in range(self.M):
            # Calculate probabilities once at start of round m for ALL classes
            probs = softmax(F)
            round_trees = []
            delta_F = np.zeros_like(F)

            # Build K separate trees using frozen round probabilities
            for k in range(self.K):
                residuals_k = Y_onehot[:, k] - probs[:, k]
                hessians_k = probs[:, k] * (1.0 - probs[:, k])

                tree = MultiClassGBCStump()
                tree.fit(X, residuals_k, hessians_k)

                gamma_k = tree.predict(X)
                delta_F[:, k] = self.nu * gamma_k

                round_trees.append(tree)

            # Synchronously update log-odds after fitting all K trees
            F += delta_F
            self.trees.append(round_trees)

            print(
                f"Round {m+1} complete. Updated Probabilities:\n{np.round(softmax(F), 3)}\n"
            )

    def predict_proba(self, X_new):
        n_samples = X_new.shape[0]
        F = np.zeros((n_samples, self.K), dtype=np.float64)

        for m in range(self.M):
            for k in range(self.K):
                tree = self.trees[m][k]
                F[:, k] += self.nu * tree.predict(X_new)

        return softmax(F)

    def predict(self, X_new):
        probs = self.predict_proba(X_new)
        return np.argmax(probs, axis=1)


if __name__ == "__main__":
    # Toy 3-Class dataset
    X_train = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])
    y_train = np.array([0, 0, 1, 1, 2, 2])

    gbc = MultiClassGradientBoostingClassifier(
        n_classes=3, n_estimators=3, learning_rate=0.2
    )
    gbc.fit(X_train, y_train)

    X_test = np.array([[1.2], [3.4], [5.8]])
    probs = gbc.predict_proba(X_test)
    preds = gbc.predict(X_test)

    print("--- INFERENCE RESULTS ---")
    for x_val, p_vals, c_val in zip(X_test.ravel(), probs, preds):
        print(
            f"X = {x_val}: Probabilities = {np.round(p_vals, 3)} -> Predicted Class = {c_val}"
        )

--- MULTI-CLASS GBC TRAINING LOOP ---
Number of Classes K = 3
Initial Probabilities p(0):
[[0.333 0.333 0.333]
 [0.333 0.333 0.333]
 [0.333 0.333 0.333]
 [0.333 0.333 0.333]
 [0.333 0.333 0.333]
 [0.333 0.333 0.333]]

Round 1 complete. Updated Probabilities:
[[0.552 0.224 0.224]
 [0.552 0.224 0.224]
 [0.28  0.44  0.28 ]
 [0.28  0.44  0.28 ]
 [0.199 0.312 0.489]
 [0.199 0.312 0.489]]

Round 2 complete. Updated Probabilities:
[[0.646 0.214 0.14 ]
 [0.646 0.214 0.14 ]
 [0.227 0.546 0.227]
 [0.227 0.546 0.227]
 [0.136 0.208 0.656]
 [0.136 0.208 0.656]]

Round 3 complete. Updated Probabilities:
[[0.762 0.144 0.094]
 [0.762 0.144 0.094]
 [0.183 0.634 0.182]
 [0.183 0.634 0.182]
 [0.086 0.19  0.723]
 [0.086 0.19  0.723]]

--- INFERENCE RESULTS ---
X = 1.2: Probabilities = [0.762 0.144 0.094] -> Predicted Class = 0
X = 3.4: Probabilities = [0.183 0.634 0.182] -> Predicted Class = 1
X = 5.8: Probabilities = [0.086 0.19  0.723] -> Predicted Class = 2
